# 05 · Cypher at scale — read-only on a ~150k-node graph

Points the cypher server (read-only) at the pre-existing `arxiv` graph (Papers/Authors/Categories). Schema sampling, big aggregations, oversized-property sanitization, and pagination all hold up. Skips cleanly if the graph isn't present.

In [1]:
import warnings; warnings.filterwarnings("ignore")   # quiet 3rd-party import warnings
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # examples/demos
from _common import clients, console
print("helpers ready — no API key needed (the MCP servers are pure tools)")

helpers ready — no API key needed (the MCP servers are pure tools)


## Schema (sampled) + an aggregation at scale

In [2]:
from _common import config
async def scale():
    if not config.graph_exists("agensgraph_demos","arxiv"):
        print("arxiv graph not present — skipping (the flights demos still run)."); return
    async with clients.cypher_client("agensgraph_demos","arxiv", read_only=True) as cy:
        schema=clients.data(await cy.call_tool("get_agensgraph_schema",{}))
        for label,info in schema.items(): console.kv(label, f"{info.get('count'):,} nodes")
        cats=clients.data(await cy.call_tool("read_agensgraph_cypher",
            {"query":'MATCH (:"Paper")-[:"IN_CATEGORY"]->(c:"Category") RETURN c.name AS category, count(*) AS papers ORDER BY papers DESC LIMIT 5'}))
        console.table([(r["category"],r["papers"]) for r in cats["rows"]], headers=["category","papers"])
        p=clients.data(await cy.call_tool("read_agensgraph_cypher",{"query":'MATCH (p:"Paper") RETURN p LIMIT 1'}))
        console.kv("Paper keys (embedding stripped)", list(p["rows"][0]["p"].keys()))
await scale()

  Paper                      55,000 nodes
  category  papers
  --------  ------
  astro-ph  11160 
  hep-ph    5228  
  hep-th    4948  
  quant-ph  3482  
  gr-qc     2952  
  Paper keys (embedding stripped) ['id', 'year', 'title', 'abstract']
